In [4]:
import pandas as pd

# Load the Student Performance dataset from the official UCI repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"

# Colab allows us to read directly from zipped files if we specify the CSV inside
import requests, zipfile, io
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

# Loading the math course data (student-mat.csv is typically used for this project)
# Note: The UCI student dataset uses semicolons (;) as separators
df = pd.read_csv(z.open('student-mat.csv'), sep=';')

print("Dataset successfully loaded into 'df'!")

Dataset successfully loaded into 'df'!


In [5]:
import pandas as pd

# Generate a clean, draft data dictionary summary table
summary = pd.DataFrame({
    "variable": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "n_unique": df.nunique().values,
    "n_missing": df.isna().sum().values,
})

display(summary)

,variable,dtype,n_unique,n_missing
0,school,object,2,0
1,sex,object,2,0
2,age,int64,8,0
3,address,object,2,0
4,famsize,object,2,0
5,Pstatus,object,2,0
6,Medu,int64,5,0
7,Fedu,int64,5,0
8,Mjob,object,5,0
9,Fjob,object,5,0


## Summary

The raw dataset contains 10 key student variables with mixed data types, including 3 categorical columns (`object`) and 7 numerical columns (`int64`). A preliminary analysis indicates that there are no missing values across any of these columns (`n_missing` is 0 for all) , ensuring a structurally complete data profile ready for feature grouping. This baseline table serves as the initial framework for our project's data dictionary.

---

## Interpretation

To convert the technical metadata into an interpretable resource, the columns are organized into four foundational feature domains based on their real-world context:

### 1. Demographic Variables

These variables outline basic background attributes that describe who the student is.

*
**`school`** (`object`): The specific educational institution the student attends, featuring 2 unique values.


*
**`sex`** (`object`): The student's biological sex, split into 2 unique categories.


*
**`age`** (`int64`): The student's age in whole numbers, spanning 8 unique values.


*
**`address`** (`object`): The location type of the student's home environment (e.g., urban vs. rural), containing 2 unique values.



### 2. Social Variables

These factors track external family connections, social habits, and environmental contexts.

*
(Note: While variables such as parent education or alcohol use are tracked in the wider program scope, no social variables are present within this specific 10-column subset.)



### 3. School-Related Variables

These profiles focus on student academic habits, behaviors, and baseline environmental support structures.

*
**`studytime`** (`int64`): A metric representing the student's weekly study time category, broken into 4 distinct groups.


*
**`absences`** (`int64`): The total count of school days missed by the student, showing high variation across 34 unique values.



### 4. Academic Variables

These columns represent direct measures of previous and concurrent academic progress or standing.

*
**`failures`** (`int64`): The number of past class or course failures, mapped across 4 unique values.


*
**`G1`** (`int64`): The student's numerical grade achieved during the first period evaluation, featuring 17 unique values.


*
**`G2`** (`int64`): The student's numerical grade achieved during the second period evaluation, featuring 16 unique values.


*
**`G3`** (`int64`): The final evaluation grade, showing 18 unique values.



---

## Recommendation

*
**Establish the Core Target:** Set `G3` as the primary prediction target for the machine learning pipeline, as it represents the absolute performance outcome.


*
**Address Target Leakage Risks:** The choice of features must depend strictly on the project's operational deployment goals. If you want to design an **early-warning system** to identify at-risk students near the beginning of the semester, you **must exclude `G1` and `G2**` from your models. Keeping them introduces leakage because midterm performance will dominate the model, masking early behavioral signals like `absences` or `studytime`.


*
**Actionability and Intervention Priorities:** Focus heavily on behavioral features (`absences`, `studytime`) and performance history (`failures`) during final feature evaluation. While demographic constraints help spotlight underlying socio-economic or structural divides , changing a student's study habits or providing focused tutoring gives schools direct, actionable paths to improve student retention and academic success.

In [6]:
df.columns

Index(['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G1', 'G2', 'G3'],
      dtype='object')

In [7]:
for col in df.columns:
  print(col)

school
sex
age
address
famsize
Pstatus
Medu
Fedu
Mjob
Fjob
reason
guardian
traveltime
studytime
failures
schoolsup
famsup
paid
activities
nursery
higher
internet
romantic
famrel
freetime
goout
Dalc
Walc
health
absences
G1
G2
G3


In [9]:
feature_groups = {
    "demographic": [
        "school", "sex", "age", "address"
    ],
    "social": [
        "famsize", "Pstatus", "Medu", "Fedu", "Mjob", "Fjob",
        "guardian", "famrel", "freetime", "goout", "Dalc", "Walc",
        "romantic"
    ],
    "school_related": [
        "reason", "traveltime", "studytime", "schoolsup", "famsup",
        "paid", "activities", "nursery", "higher", "internet", "absences"
    ],
    "academic": [
        "failures", "G1", "G2", "G3"
    ]
}

feature_groups

{'demographic': ['school', 'sex', 'age', 'address'],
 'social': ['famsize',
  'Pstatus',
  'Medu',
  'Fedu',
  'Mjob',
  'Fjob',
  'guardian',
  'famrel',
  'freetime',
  'goout',
  'Dalc',
  'Walc',
  'romantic'],
 'school_related': ['reason',
  'traveltime',
  'studytime',
  'schoolsup',
  'famsup',
  'paid',
  'activities',
  'nursery',
  'higher',
  'internet',
  'absences'],
 'academic': ['failures', 'G1', 'G2', 'G3']}

In [12]:
grouped_columns = []
for group, columns in feature_groups.items():
  grouped_columns.extend(columns)
missing_from_groups = set(df.columns) - set(grouped_columns)
extra_in_groups = set(grouped_columns) - set(df.columns)
print("Columns not assigned to any group:")
print(missing_from_groups)
print("\nColumns listed in groups but not found in dataset:")
print(extra_in_groups)


Columns not assigned to any group:
{'health'}

Columns listed in groups but not found in dataset:
set()


A completed column-group table that assigns every dataset variable to one of four
groups:
demographic, social, school-related, or academic.

In [13]:
file_name = "data_dictionary.md"
print(f"Ready to create: {file_name}")

Ready to create: data_dictionary.md


Demographic Variables
## 1. Demographic Variables
| Variable | Description | Data Type | Missing Values |
|---|---|---:|---:|
| school | Student's school. | object | 0 |
| sex | Student's sex. | object | 0 |
| age | Student's age. | int64 | 0 |
| address | Student's home address type, such as urban or rural. | object | 0 |
Social Variables
## 2. Social Variables
| Variable | Description | Data Type | Missing Values |
|---|---|---:|---:|
| famsize | Student's family size category. | object | 0 |
| Pstatus | Parent cohabitation status. | object | 0 |
| Medu | Mother's education level. | int64 | 0 |
| Fedu | Father's education level. | int64 | 0 |
| Mjob | Mother's job category. | object | 0 |
| Fjob | Father's job category. | object | 0 |
| guardian | Student's guardian. | object | 0 |
| famrel | Quality of family relationships. | int64 | 0 |
| freetime | Student's free time after school. | int64 | 0 |
| goout | Frequency of going out with friends. | int64 | 0 |
| Dalc | Workday alcohol consumption. | int64 | 0 |
| Walc | Weekend alcohol consumption. | int64 | 0 |
| romantic | Whether the Student is in a romantic relationship. | object | 0 |
School-Related Variables
## 3. School-Related Variables
| Variable | Description | Data Type | Missing Values |
|---|---|---:|---:|
| reason | Reason for choosing the school. | object | 0 |
| traveltime | Travel time from home to school. | int64 | 0 |
| studytime | Weekly study time category. | int64 | 0 |
| schoolsup | Whether the Student receives extra school support. | object | 0 |
| famsup | Whether the Student receives family educational support. | object | 0 |
| paid | Whether the Student receives extra paid classes. | object | 0 |
| activities | Whether the Student participates in extracurricular activities. |
object | 0 |
| nursery | Whether the Student attended nursery school. | object | 0 |
| higher | Whether the Student wants to pursue higher education. | object | 0 |

GSSRP 2026 - Kean University | Predicting Student Performance Using Machine Learning Session 7/48 - Week 1
| internet | Whether the Student has internet access at home. | object | 0 |
| absences | Number of school absences. | int64 | 0 |
Academic Variables
## 4. Academic Variables
| Variable | Description | Data Type | Missing Values |
|---|---|---:|---:|
| failures | Number of past class failures. | int64 | 0 |
| G1 | First-period grade. | int64 | 0 |
| G2 | Second-period grade. | int64 | 0 |
| G3 | Final grade. This is usually the main prediction target. | int64 | 0 |

In [14]:
import os
import shutil
from google.colab import files

# 1. Define the markdown text contents
data_dictionary_text = """
# Data Dictionary

## Project
Student Performance Prediction Using Machine Learning

## Purpose
This data dictionary describes the variables used in the student performance dataset. The variables are grouped into demographic, social, school-related, and academic feature groups.

## Summary
The dataset includes student background information, family and social information, school-related behavior, and academic performance variables. The final grade variable, G3, can serve as the primary prediction target.

## 1. Demographic Variables

| Variable | Description | Data Type | Missing Values |
|---|---|---|---|
| school | Student's school. | object | 0 |
| sex | Student's sex. | object | 0 |
| age | Student's age. | int64 | 0 |
| address | Student's home address type, such as urban or rural. | object | 0 |

## 2. Social Variables

| Variable | Description | Data Type | Missing Values |
|---|---|---|---|
| famsize | Student's family size category. | object | 0 |
| Pstatus | Parent cohabitation status. | object | 0 |
| Medu | Mother's education level. | int64 | 0 |
| Fedu | Father's education level. | int64 | 0 |
| Mjob | Mother's job category. | object | 0 |
| Fjob | Father's job category. | object | 0 |
| guardian | Student's guardian. | object | 0 |
| famrel | Quality of family relationships. | int64 | 0 |
| freetime | Student's free time after school. | int64 | 0 |
| goout | Frequency of going out with friends. | int64 | 0 |
| Dalc | Workday alcohol consumption. | int64 | 0 |
| Walc | Weekend alcohol consumption. | int64 | 0 |
| romantic | Whether the student is in a romantic relationship. | object | 0 |

## 3. School-Related Variables

| Variable | Description | Data Type | Missing Values |
|---|---|---|---|
| reason | Reason for choosing the school. | object | 0 |
| traveltime | Travel time from home to school. | int64 | 0 |
| studytime | Weekly study time category. | int64 | 0 |
| schoolsup | Whether the student receives extra school support. | object | 0 |
| famsup | Whether the student receives family educational support. | object | 0 |
| paid | Whether the student receives extra paid classes. | object | 0 |
| activities | Whether the student participates in extracurricular activities. | object | 0 |
| nursery | Whether the student attended nursery school. | object | 0 |
| higher | Whether the student wants to pursue higher education. | object | 0 |
| internet | Whether the student has internet access at home. | object | 0 |
| absences | Number of school absences. | int64 | 0 |

## 4. Academic Variables

| Variable | Description | Data Type | Missing Values |
|---|---|---|---|
| failures | Number of past class failures. | int64 | 0 |
| G1 | First-period grade. | int64 | 0 |
| G2 | Second-period grade. | int64 | 0 |
| G3 | Final grade. This is usually the main prediction target. | int64 | 0 |

## Interpretation Notes
The demographic variables describe basic student background information. The social variables include family structure, parents' education, parents' occupations, social behavior, and home environment. The school-related variables describe study habits, support systems, attendance, and school engagement. The academic variables describe previous academic performance. G3 is the final grade and can serve as the primary target variable for prediction.

## Modeling Note
If the project goal is full-information prediction, G1 and G2 may be included as predictors of G3. If the project goal is early-warning prediction, G1 and G2 should be excluded because they may introduce target leakage. In that case, the model should use only information available early in the semester.
"""

# 2. Programmatically create the file locally in Colab
with open("data_dictionary.md", "w", encoding="utf-8") as file:
    file.write(data_dictionary_text.strip())

# 3. Create the reports directory inside Colab
os.makedirs("reports", exist_ok=True)

# 4. Safely move the file into the reports folder
shutil.move("data_dictionary.md", "reports/data_dictionary.md")
print("Successfully built reports/data_dictionary.md inside Colab!")

# 5. Automatically trigger a browser download to your Mac
files.download("reports/data_dictionary.md")

Successfully built reports/data_dictionary.md inside Colab!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>